# 00 · Colab timing: how long is one epoch on a free T4?

**Owner:** Experimentation Engineer · **Phase 2**

The Phase 3 studies need many runs, and a free Colab session is limited. This notebook measures
**seconds per training epoch** for the full train split (**34,633 images, batch 32 → 1,083 steps**) on a free
Colab **T4**, for:

| # | Model | Image size | Weight storage |
|---|---|---|---|
| 1 | ResNet-50, last block (conv5) unfrozen | 224 px | fp32 |
| 2 | ResNet-50, last block unfrozen | 224 px | bf16_sr |
| 3 | ResNet-50, last block unfrozen | 160 px | bf16_sr |
| 4 | Model A (design in `report/model_a_design.md`) | 224 px | fp32 |

**Method.** Real data through the team pipeline (`src.data.load_plantvillage`, augmentation on), so decoding and
augmentation are part of the time. For each setting: 10 warm-up steps (graph tracing, cuDNN autotuning), then
100 timed steps; the time per step × 1,083 gives the epoch time. One pass over the validation split (8,659 images)
is timed the same way. The models are built here only for timing; the real builders come in `src/models.py`.

## 1. Runtime and repository

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/saperaa/frostleaf.git
%cd frostleaf
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Data: PlantVillage colour with the fixed leaf-grouped split

In [ ]:
!python data/download.py --variants color --skip-plantdoc

In [ ]:
import time, math, numpy as np, keras, tensorflow as tf
from src.config import load_config
from src.data import load_plantvillage
from src.precision import PrecisionModel

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "|", tf.config.list_physical_devices("GPU"))
BATCH, TRAIN_IMAGES, VAL_IMAGES = 32, 34_633, 8_659
STEPS_PER_EPOCH = math.ceil(TRAIN_IMAGES / BATCH)   # 1,083
VAL_STEPS = math.ceil(VAL_IMAGES / BATCH)           # 271
WARMUP, TIMED = 10, 100

def datasets(image_size):
    cfg = load_config("base.yaml")
    cfg["data"]["image_size"] = image_size
    cfg["train"]["batch_size"] = BATCH
    train_ds, val_ds, _, class_names = load_plantvillage("color", cfg)
    return train_ds, val_ds, class_names

## 3. The models (timing versions)

In [ ]:
def resnet50_last_block(image_size, precision, num_classes=38):
    keras.utils.set_random_seed(42)
    base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                       input_shape=(image_size, image_size, 3))
    for layer in base.layers:   # only conv5 trains; BatchNorm stays frozen
        layer.trainable = layer.name.startswith("conv5_") and not isinstance(layer, keras.layers.BatchNormalization)
    inputs = keras.Input((image_size, image_size, 3))
    x = keras.applications.resnet50.preprocess_input(inputs * 255.0)   # pipeline gives [0, 1]
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return PrecisionModel(inputs, outputs, precision=precision, rounding_seed=42)

def model_a(image_size, precision, num_classes=38):
    keras.utils.set_random_seed(42)
    inputs = keras.Input((image_size, image_size, 3))
    x = inputs   # report/model_a_design.md rescales 0-255 inside; the pipeline already gives [0, 1]
    for filters in (32, 64, 128, 256):
        x = keras.layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return PrecisionModel(inputs, outputs, precision=precision, rounding_seed=42)

## 4. Timing

In [ ]:
class StepTimer(keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch == WARMUP - 1:
            self.t0 = time.perf_counter()
        if batch == WARMUP + TIMED - 1:
            self.seconds = time.perf_counter() - self.t0

def time_setting(name, builder, image_size, precision, lr):
    train_ds, val_ds, _ = datasets(image_size)
    model = builder(image_size, precision)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy")
    model.quantize_weights()
    timer = StepTimer()
    model.fit(train_ds.take(WARMUP + TIMED), epochs=1, verbose=0, callbacks=[timer])
    per_step = timer.seconds / TIMED
    model.evaluate(val_ds.take(WARMUP), verbose=0)                    # warm up evaluation
    t0 = time.perf_counter(); model.evaluate(val_ds.take(50), verbose=0)
    val_per_step = (time.perf_counter() - t0) / 50
    epoch_s = per_step * STEPS_PER_EPOCH + val_per_step * VAL_STEPS
    row = dict(setting=name, size=image_size, precision=precision,
               trainable=int(sum(np.prod(v.shape) for v in model.trainable_variables)),
               s_per_step=round(per_step, 4), train_s=round(per_step * STEPS_PER_EPOCH),
               val_s=round(val_per_step * VAL_STEPS), epoch_s=round(epoch_s),
               epochs_in_30min=int(1800 // epoch_s))
    print(row)
    keras.backend.clear_session()
    return row

results = []
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 224, "fp32", 1e-5))
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 224, "bf16_sr", 1e-5))
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 160, "bf16_sr", 1e-5))
results.append(time_setting("Model A", model_a, 224, "fp32", 1e-3))

## 5. Results

In [ ]:
import pandas as pd
table = pd.DataFrame(results)
table

## 6. Recommendation

The cell below states it from the measured numbers: the image size to use and the largest number of epochs
that fits a 30-minute Colab budget per run (training + one validation pass per epoch).

In [ ]:
c224, c224sr, c160sr, a224 = results
print(f"Model C (ResNet-50, bf16_sr): 224 px = {c224sr['epoch_s']} s/epoch, 160 px = {c160sr['epoch_s']} s/epoch "
      f"({c224sr['epoch_s'] / c160sr['epoch_s']:.2f}x faster at 160 px).")
print(f"bf16_sr vs fp32 at 224 px: {c224sr['epoch_s'] / c224['epoch_s']:.2f}x the fp32 time (the simulated rounding costs this much).")
size = 224 if c224sr["epochs_in_30min"] >= 5 else 160
chosen = c224sr if size == 224 else c160sr
print(f"Recommendation: image size {size} px; at most {chosen['epochs_in_30min']} epochs per 30-minute run for Model C "
      f"(bf16_sr), {c224['epochs_in_30min']} for Model C in fp32 at 224 px, {a224['epochs_in_30min']} for Model A at 224 px.")